# Pipeline 3 — GRPO Training Data Pipeline

**GRPO (Group Relative Policy Optimization)** — a reward-model-free RL method where the reward signal comes from **comparing multiple generated responses per prompt** rather than a trained reward model.

| Step | Mode | Produces | Purpose |
|------|------|----------|---------|
| 1 | `sft` | `prompts.bin` + `responses.bin` | SFT warm-up |
| 2 | `rollout` | `prompts.bin` | Generate G responses per prompt (group sampling) |

> **Key idea:** For each prompt, sample G responses from the policy. Score all G with a rule-based/verifiable reward. Use the group's mean as baseline — no separate reward model needed.

Repo: https://github.com/SniAssia/Optimized_data_loaders/tree/main/reinforcement_learning_data_loader

## 0. Config

In [ ]:
import os

REPO_URL      = "https://github.com/SniAssia/Optimized_data_loaders.git"
REPO_DIR      = "Optimized_data_loaders"
RL_DIR        = os.path.join(REPO_DIR, "reinforcement_learning_data_loader")
LIBTORCH_PATH = "/content/libtorch"

# GRPO group size: number of responses sampled per prompt
GRPO_GROUP_SIZE = 8  # G in the paper — adjust as needed

## 1. Clone / Pull Repo

In [ ]:
import subprocess

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

print("Repo contents:")
for f in sorted(os.listdir(RL_DIR)):
    print(" ", f)

## 2. LibTorch Setup

In [ ]:
import torch
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)

if not os.path.isdir(LIBTORCH_PATH):
    !wget -q https://download.pytorch.org/libtorch/nightly/cu128/libtorch-cxx11-abi-shared-with-deps-latest.zip
    !unzip -q libtorch-cxx11-abi-shared-with-deps-latest.zip
    print("libtorch ready")
else:
    print("libtorch already at", LIBTORCH_PATH)

## 3. Download Dataset

In [ ]:
jsonl_path = os.path.join(RL_DIR, "hh_rlhf_grpo.jsonl")

result = subprocess.run(
    [
        "python3", "prepare_dataset.py",
        "--dataset", "Anthropic/hh-rlhf",
        "--output",  "hh_rlhf_grpo.jsonl",
    ],
    cwd=RL_DIR, capture_output=True, text=True,
)
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr[-2000:] if result.stderr else "")
result.check_returncode()

## 4. Build C++ Binary

In [ ]:
import shutil

have_cmake  = shutil.which("cmake") is not None
have_gpp    = shutil.which("g++")   is not None or shutil.which("clang++") is not None
libtorch_ok = LIBTORCH_PATH and os.path.isdir(LIBTORCH_PATH)
can_build   = have_cmake and have_gpp and libtorch_ok

build_dir = os.path.join(RL_DIR, "build")
build_ok  = False

if can_build:
    os.makedirs(build_dir, exist_ok=True)
    cfg = subprocess.run(
        ["cmake", f"-DCMAKE_PREFIX_PATH={LIBTORCH_PATH}", ".."],
        cwd=build_dir, capture_output=True, text=True,
    )
    if cfg.returncode == 0:
        bld = subprocess.run(
            ["cmake", "--build", ".", "-j"],
            cwd=build_dir, capture_output=True, text=True,
        )
        print(bld.stdout[-2000:])
        build_ok = bld.returncode == 0
    else:
        print(cfg.stderr[-1000:])

print("build_ok =", build_ok)
binary_path = os.path.join(build_dir, "dataloader_demo")

---
# GRPO Phase 1 — SFT Warm-Up Batches

**Why SFT before GRPO?**
GRPO samples G responses per prompt. If the model is completely untrained, all G responses are garbage and the group reward signal is meaningless. SFT gives the model a sensible starting point.

**Batch format:** `(prompt, response, labels)`

In [ ]:
sft_out = os.path.join(RL_DIR, "out_grpo_sft")

result = subprocess.run(
    [
        "python3", "tokenize_dataset.py",
        "--input",      "hh_rlhf_grpo.jsonl",
        "--output-dir", "out_grpo_sft",
        "--mode",       "sft",
    ],
    cwd=RL_DIR, capture_output=True, text=True,
)
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr[-1500:] if result.stderr else "")
result.check_returncode()

In [ ]:
if build_ok and os.path.isfile(binary_path):
    print("========== GRPO Phase 1: SFT Warm-Up ==========")
    run = subprocess.run(
        [os.path.abspath(binary_path), "sft"],
        cwd=sft_out, capture_output=True, text=True,
    )
    print(run.stdout)
    if run.returncode != 0:
        print("STDERR:", run.stderr)
else:
    print("Binary not built.")

### SFT Batch Shape
```
[SFT] batch 0  prompt=[4, 128]  response=[4, 128]  labels=[4, 128]
```
Train until loss converges, then move to the GRPO rollout phase.

---
# GRPO Phase 2 — Rollout Batches (Group Sampling)

**This is the core of GRPO.** The dataloader provides **only prompts**. The policy then generates **G responses per prompt** at inference time.

```
prompt_i  →  [response_1, response_2, ..., response_G]   (G samples from policy)
                    ↓                          ↓
               reward_1               reward_G      (rule-based or verifier)
                    ↓
          advantage_i = reward_i - mean(reward_1..G)   (group relative baseline)
```

**No reward model trained separately** — the reward comes from a verifiable signal
(e.g. math answer correctness, code execution, format checking).

**Files produced:** `prompts.bin` only — responses are generated online.

In [ ]:
rollout_out = os.path.join(RL_DIR, "out_grpo_rollout")

result = subprocess.run(
    [
        "python3", "tokenize_dataset.py",
        "--input",      "hh_rlhf_grpo.jsonl",
        "--output-dir", "out_grpo_rollout",
        "--mode",       "rollout",
    ],
    cwd=RL_DIR, capture_output=True, text=True,
)
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr[-1500:] if result.stderr else "")
result.check_returncode()

In [ ]:
fpath = os.path.join(rollout_out, "prompts.bin")
if os.path.isfile(fpath):
    print(f"prompts.bin: {os.path.getsize(fpath)/1e6:.2f} MB")
else:
    print("prompts.bin: NOT FOUND")

In [ ]:
if build_ok and os.path.isfile(binary_path):
    print("========== GRPO Phase 2: Rollout (Group Sampling) ==========")
    run = subprocess.run(
        [os.path.abspath(binary_path), "rollout"],
        cwd=rollout_out, capture_output=True, text=True,
    )
    print(run.stdout)
    if run.returncode != 0:
        print("STDERR:", run.stderr)
else:
    print("Binary not built.")

### What Rollout Batches Look Like for GRPO

```
[ROLLOUT] batch 0  prompt=[4, 64]
                   ↑ only prompts
```

At training time, your GRPO loop expands each prompt to G copies:

```python
# Pseudo-code of what happens AFTER the dataloader
prompts_expanded = prompts.repeat_interleave(G, dim=0)  # [B*G, T]
responses        = policy.generate(prompts_expanded)     # [B*G, T_resp]
rewards          = reward_fn(prompts_expanded, responses) # [B*G]
rewards_grouped  = rewards.view(B, G)                    # [B, G]
advantages       = rewards_grouped - rewards_grouped.mean(dim=1, keepdim=True)  # group baseline
loss             = -log_prob(responses) * advantages     # GRPO loss
```

### Effect of Group Size G

| G | Advantage | Disadvantage |
|---|-----------|-------------|
| Small (4) | Fast, less memory | Noisy baseline |
| Large (16+) | Stable baseline | More compute per step |

Typical values: G = 8 to 16 in practice (DeepSeek-R1 uses G=8).

Your configured group size: **G = {GRPO_GROUP_SIZE}** (set in Config cell)

### Verify Group Sampling Logic

In [ ]:
import struct
from transformers import AutoTokenizer

def read_bin(path):
    seqs = []
    with open(path, "rb") as f:
        (n,) = struct.unpack("<I", f.read(4))
        for _ in range(n):
            (length,) = struct.unpack("<H", f.read(2))
            ids = struct.unpack(f"<{length}I", f.read(4 * length))
            seqs.append(list(ids))
    return seqs

prompts = read_bin(os.path.join(rollout_out, "prompts.bin"))
print(f"Total prompts loaded: {len(prompts)}")
print(f"With GRPO_GROUP_SIZE={GRPO_GROUP_SIZE}, each training step uses {GRPO_GROUP_SIZE} responses per prompt")
print(f"Effective batch size for rollout: batch_size x G = 4 x {GRPO_GROUP_SIZE} = {4 * GRPO_GROUP_SIZE} sequences")

tok = AutoTokenizer.from_pretrained("inceptionai/jais-family-590m", trust_remote_code=True)
print("\n=== Sample prompt (decoded) ===")
print(tok.decode(prompts[0])[:300])

---
# Full GRPO Pipeline Summary

```
hh_rlhf_grpo.jsonl
       │
       ├─── tokenize (sft)     → out_grpo_sft/prompts.bin + responses.bin
       │         ↓
       │    [SFT batch]  prompt=[B,T]  response=[B,T]  labels=[B,T]
       │    → warm-up: policy learns basic instruction following
       │
       └─── tokenize (rollout) → out_grpo_rollout/prompts.bin
                 ↓
            [Rollout batch]  prompt=[B,T]
                 ↓  (expand by G in training loop)
            prompt × G  →  G responses generated by policy
                 ↓
            rule-based reward scores all G responses
                 ↓
            advantage = reward - group_mean  (no reward model!)
                 ↓
            GRPO loss updates policy
```

### GRPO vs PPO vs DPO

| | PPO | DPO | GRPO |
|--|-----|-----|------|
| Reward model | ✅ trained | ❌ | ❌ |
| Preference pairs | ✅ | ✅ | ❌ |
| Online generation | ✅ | ❌ | ✅ |
| Baseline | value function | implicit | group mean reward |
| Best for | general RLHF | preference data | verifiable tasks (math, code) |